In [0]:
%run "/Workspace/Users/iancarodrigues108@gmail.com/ETL_Arquitetura_Medalhao_BOI_GORDO/0.Config/config"




catalogo:  workspace
schemas:  bronze_economia silver_economia gold_economia


# Ingestao da  IPCA na nossa camada Bronze utilizando o API do banco cental


In [0]:
#Explicando nessas fórmulas utilizando o url do site da BCB da serie 433 para o ipca formatando a data inicial e a data final para o formato brasileiro e o df_pd para o dataframe pandas data e valor e o ipca str replace para substituir a virgula por ponto e astype para converter para float

import requests, pandas as pd
from pyspark.sql.functions import current_timestamp

url= "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
params = {"formato": "json", "dataInicial": "01/01/2025", "dataFinal": "31/12/2025"}

df_pd = pd.DataFrame(requests.get(url, params=params).json())
df_pd.columns = ["data", "ipca"]
df_pd["ipca"] = df_pd["ipca"].str.replace(",",".").astype(float)

In [0]:
df_pd.display()

data,ipca
01/01/2025,0.16
01/02/2025,1.31
01/03/2025,0.56
01/04/2025,0.43
01/05/2025,0.26
01/06/2025,0.24
01/07/2025,0.26
01/08/2025,-0.11
01/09/2025,0.48
01/10/2025,0.09


In [0]:
df_pd = spark.createDataFrame(df_pd).withColumn("data_coleta", current_timestamp())

In [0]:
df_pd.display()

data,ipca,data_coleta
01/01/2025,0.16,2026-07-22T17:48:49.297Z
01/02/2025,1.31,2026-07-22T17:48:49.297Z
01/03/2025,0.56,2026-07-22T17:48:49.297Z
01/04/2025,0.43,2026-07-22T17:48:49.297Z
01/05/2025,0.26,2026-07-22T17:48:49.297Z
01/06/2025,0.24,2026-07-22T17:48:49.297Z
01/07/2025,0.26,2026-07-22T17:48:49.297Z
01/08/2025,-0.11,2026-07-22T17:48:49.297Z
01/09/2025,0.48,2026-07-22T17:48:49.297Z
01/10/2025,0.09,2026-07-22T17:48:49.297Z


In [0]:
df_pd.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE}.ipca")
